In [1]:
# Clone the repo
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USER = "ThemiyaDeshan"
REPO_NAME = "Y03S01_Machine-Lerning_Assignment"

repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"

!git clone {repo_url}
!git -C {REPO_NAME} config user.email "deshanxd@gmail.com"
!git -C {REPO_NAME} config user.name "ThemiyaDeshan"

Cloning into 'Y03S01_Machine-Lerning_Assignment'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 30 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 579.35 KiB | 6.90 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [2]:
# Load the raw data
!pip install -q ucimlrepo
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

bank_marketing = fetch_ucirepo(id=222)
X = bank_marketing.data.features
y = bank_marketing.data.targets
df = pd.concat([X, y], axis=1)
print(df.shape)

(45211, 17)


In [3]:
# Handle missing/unknown categories
# job, education, contact: genuine missingness -> explicit "unknown" category
for col in ["job", "education", "contact"]:
    df[col] = df[col].fillna("unknown")

# poutcome: STRUCTURAL missingness (only missing when never contacted before)
# -> its own meaningful category, not "unknown"
df["poutcome"] = df["poutcome"].fillna("no_prior_campaign")

print(df[["job", "education", "contact", "poutcome"]].isnull().sum())

job          0
education    0
contact      0
poutcome     0
dtype: int64


In [4]:
# Feature engineering for the pdays sentinel
# Binary flag: was this client contacted in a previous campaign at all?
df["previously_contacted"] = (df["pdays"] != -1).astype(int)

# Recode the -1 sentinel to a large value ("very long since contact") instead of
# leaving a fake negative number that would distort scaling/distance calculations
df["pdays_capped"] = df["pdays"].replace(-1, df["pdays"].max() + 1)

df[["pdays", "previously_contacted", "pdays_capped"]].describe()

,pdays,previously_contacted,pdays_capped
count,45211.000000,45211.000000,45211.000000
mean,40.197828,0.182633,753.759616
std,100.128746,0.386369,254.954125
min,-1.000000,0.000000,1.000000
25%,-1.000000,0.000000,872.000000
50%,-1.000000,0.000000,872.000000
75%,-1.000000,0.000000,872.000000
max,871.000000,1.000000,872.000000


In [5]:
# Remove the leakage variable
df_diagnostic = df.copy()          # keeps duration, for a diagnostic-only benchmark model later
df = df.drop(columns=["duration", "pdays"])   # raw pdays replaced by pdays_capped + previously_contacted

print(df.columns.tolist())

['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day_of_week', 'month', 'campaign', 'previous', 'poutcome', 'y', 'previously_contacted', 'pdays_capped']


In [6]:
# Encode target and categorical features
df["y"] = df["y"].map({"no": 0, "yes": 1})

categorical_cols = ["job", "marital", "education", "default", "housing",
                     "loan", "contact", "month", "poutcome"]
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(df_encoded.shape)
df_encoded.head()

(45211, 43)


,age,balance,day_of_week,campaign,previous,y,previously_contacted,pdays_capped,job_blue-collar,job_entrepreneur,...,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_no_prior_campaign,poutcome_other,poutcome_success
0,58,2143,5,1,0,0,0,872,False,False,...,False,False,False,True,False,False,False,True,False,False
1,44,29,5,1,0,0,0,872,False,False,...,False,False,False,True,False,False,False,True,False,False
2,33,2,5,1,0,0,0,872,False,True,...,False,False,False,True,False,False,False,True,False,False
3,47,1506,5,1,0,0,0,872,True,False,...,False,False,False,True,False,False,False,True,False,False
4,33,1,5,1,0,0,0,872,False,False,...,False,False,False,True,False,False,False,True,False,False


In [7]:
# Stratified train/test split
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=["y"])
y_target = df_encoded["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_target, test_size=0.2, stratify=y_target, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))

(36168, 42) (9043, 42)
y
0    0.883018
1    0.116982
Name: proportion, dtype: float64


In [8]:
# Scale numeric features
from sklearn.preprocessing import RobustScaler

numeric_cols = ["age", "balance", "day_of_week", "campaign", "previous"]

X_train = X_train.drop(columns=["pdays_capped"])
X_test = X_test.drop(columns=["pdays_capped"])

scaler = RobustScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train_scaled[numeric_cols].describe()

,age,balance,day_of_week,campaign,previous
count,36168.000000,36168.000000,36168.000000,36168.000000,36168.000000
mean,0.126200,0.674281,-0.014003,0.381967,0.581730
std,0.708472,2.262521,0.640922,1.552080,2.408766
min,-1.400000,-6.245161,-1.153846,-0.500000,0.000000
25%,-0.400000,-0.277972,-0.615385,-0.500000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.600000,0.722028,0.384615,0.500000,0.000000
max,3.733333,74.968479,1.153846,30.500000,275.000000


In [9]:
# Assemble final files
train_final = X_train_scaled.copy()
train_final["y"] = y_train.values

test_final = X_test_scaled.copy()
test_final["y"] = y_test.values

print(train_final.shape, test_final.shape)

(36168, 42) (9043, 42)


In [10]:
# Save into Data/PreprocessedData and push to GitHub
import os

output_dir = f"/content/{REPO_NAME}/Data/PreprocessedData"
os.makedirs(output_dir, exist_ok=True)

train_final.to_csv(f"{output_dir}/bank_train.csv", index=False)
test_final.to_csv(f"{output_dir}/bank_test.csv", index=False)
df.to_csv(f"{output_dir}/bank_cleaned.csv", index=False)   # human-readable, pre-encoding version

print(os.listdir(output_dir))

['bank_cleaned.csv', 'PreprocessedData.txt', 'bank_test.csv', 'bank_train.csv']


In [11]:
os.chdir(f"/content/{REPO_NAME}")
!git pull origin main --no-rebase
!git add Data/PreprocessedData
!git commit -m "Add preprocessed train/test data and cleaned dataset"
!git push

From https://github.com/ThemiyaDeshan/Y03S01_Machine-Lerning_Assignment
 * branch            main       -> FETCH_HEAD
Already up to date.
[main 22c4a83] Add preprocessed train/test data and cleaned dataset
 3 files changed, 90425 insertions(+)
 create mode 100644 Data/PreprocessedData/bank_cleaned.csv
 create mode 100644 Data/PreprocessedData/bank_test.csv
 create mode 100644 Data/PreprocessedData/bank_train.csv
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 1.13 MiB | 1.82 MiB/s, done.
Total 7 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/ThemiyaDeshan/Y03S01_Machine-Lerning_Assignment.git
   ef1a859..22c4a83  main -> main
